# 三種不同的模型的資料前處理

## 共同部分
### 切割資料集
請固定隨機變數以利重現

### 刪除冗餘變數
在 02_data_quality.inpynb 所提到的變數

## 1. 隨機森林 (Random Forest)

隨機森林是由多棵決策樹透過 Bagging 組成的集成模型。它的特性是「基於規則切割」，不計算空間距離。

### 特徵縮放 (Feature Scaling)：不需要 ❌
無論月收入是 5,000 還是 0.5，決策樹尋找最佳切割點（例如：月收入 < 4000 則違約率提升）的結果完全一樣。不需要標準化或常態化。

### 離群值處理 (Outlier Handling)：完全保留，不需處理 ❌
樹狀模型對離群值極度免疫。極端的 debt_to_income_ratio (DTI) 只會被孤立到樹的某個特定葉節點中，不會去拉扯整體的迴歸線。這對抓取違約特徵非常有利。

### 類別變數編碼 (Categorical Encoding)： One-Hot Encoding ✅
若使用 scikit-learn，通常建議轉為 One-Hot Encoding 避免模型誤判數字大小有意義；但在隨機森林中，即使使用數字編碼，因為它是切分節點，影響相對線性模型小很多。

## 2. 梯度提升樹 (XGBoost / LightGBM)

這是目前在 Kaggle 表格型資料中最常勝的演算法。由多棵決策樹透過 Boosting（迭代修正錯誤）組成。

### 類別變數編碼 (Categorical Encoding)：One-Hot Encoding 
✅使用 One-Hot Encoding。

### 離群值處理 (Outlier Handling)：基本不需處理 ❌

雖然梯度提升在計算殘差 (Residuals) 時，極端離群值可能會產生較大的梯度，但一般而言 XGBoost / LightGBM 仍非常穩健。建議保留所有真實的極端值（如高 App 使用頻率、高 DTI）。

## 3. 深度神經網路 (Deep Neural Networks / MLP)

神經網路透過反向傳播 (Backpropagation) 與梯度下降法 (Gradient Descent) 更新權重。它對資料的數值分佈與尺度極度敏感。

### 特徵縮放 (Feature Scaling)：絕對必要 🚨

如果不縮放，monthly_income (幾千幾萬) 會在第一層神經元就引發梯度爆炸，或完全蓋過 bnpl_installments (3~12) 的影響。

建議：使用 StandardScaler 將所有數值特徵轉為平均值 0、標準差 1。

### 類別變數編碼 (Categorical Encoding)：One-Hot Encoding 或 Embeddings 🚨

絕對不能用 0,1,2,3 的 Label Encoding，神經網路會認為 3 是 1 的三倍大。

### 離群值處理 (Outlier Handling)：必須處理 🚨

神經網路非常怕離群值。一個極端的離群值會產生巨大的誤差 (Loss)，導致整個網路權重被拉偏。

處理建議 1：對數轉換 (Log Transformation)：先取 np.log1p(x) 讓其接近常態，再做 StandardScaler。

處理建議 2：Winsorization (截斷)：五種有離群值找出 Training set 的 99% 分位數，將所有大於該分位數的值，強制覆蓋為 99% 分位數的值。這能在不刪除資料的前提下，消除離群值對神經網路的破壞性影響。